# 04 - Evaluate answers (numeric match + LLM-as-judge)

Score the table-Q&A system against the frozen benchmark in `eval/gold_generation.jsonl`. Numeric/factual checks run locally; Gemini then judges correctness and faithfulness for all answers in **one additional call**.

The full notebook makes six answer-generation calls plus one judge call. 
To run the local scoring only, set `RUN_LLM_JUDGE = False`. The gold set explicitly states handling for guests, returns, gross sales, and net CLV.

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.evaluation import (
    build_judge_prompt,
    load_gold_set,
    parse_judge_response,
    score_answer,
)
from src.gemini import (
    build_gemini_llm,
    build_pandas_query_engine,
    load_project_env,
)

load_project_env(ROOT / ".env")
print(f"Repo root: {ROOT}")
print(f"Gemini API key present: {bool(os.getenv('GOOGLE_API_KEY'))}")


Repo root: /Users/maato4/Library/CloudStorage/OneDrive-LeviStrauss&Co/Documents/GitHub/agentic_ai_learning/retail-analytics-qa
Gemini API key present: True


## Load the same cleaned data and frozen gold set

The gold targets are fixed in the JSONL file so this notebook and the naive-RAG comparison can use identical questions and ground truth.

In [2]:
data_path = ROOT / "data" / "processed" / "online_retail_2009_2010.pkl"
if not data_path.is_file():
    raise FileNotFoundError(f"Processed dataset not found: {data_path}")

df = pd.read_pickle(data_path)
gold_set = load_gold_set(ROOT / "eval" / "gold_generation.jsonl")
print("dataset shape:", df.shape)
print("benchmark questions:", len(gold_set))
display(pd.DataFrame([{
    "id": item["id"],
    "question": item["question"],
    "expected answer": item["expected_display"],
    "calculation": item["notes"],
} for item in gold_set]))


dataset shape: (525461, 14)
benchmark questions: 6


,id,question,expected answer,calculation
0,top_gross_customer,Which identified customer generated the most g...,18102,"Sum Revenue for positive-quantity, positive-pr..."
1,average_net_clv,What was the average net customer lifetime val...,"1,950.56",Identify customers with at least one valid pos...
2,top_country_revenue,Which country generated the most net revenue a...,United Kingdom,"Sum Revenue by Country across all rows, includ..."
3,largest_line_revenue,What was the largest single positive line reve...,"25,111.09",Take the maximum of Revenue (Quantity multipli...
4,identified_customer_count,How many identified customers made at least on...,"4,312",Count unique non-missing Customer IDs with at ...
5,total_net_revenue,What was total net revenue across all transact...,"9,539,484.63",Sum Revenue across every row. Negative return/...


## Generate answers and score numeric/factual matches

The query engine produces the answers; the numeric score checks every expected number within the gold set tolerance and checks required text such as the top country. This score is deterministic and independent of the judge.

In [3]:
query_engine = build_pandas_query_engine(
    df,
    model=os.getenv("GEMINI_CHAT_MODEL", "gemini-3.5-flash-lite"),
    temperature=0.0,
    max_tokens=256,
    verbose=False,
    head=20,
)

evaluation_results = []
for item in gold_set:
    response = await query_engine.aquery(item["question"])
    answer = str(response.response)
    evaluation_results.append({
        "gold": item,
        "answer": answer,
        "numeric_score": score_answer(answer, item),
    })

score_table = pd.DataFrame([{
    "id": result["gold"]["id"],
    "question": result["gold"]["question"],
    "answer": result["answer"],
    "ground truth": result["gold"]["expected_display"],
    "numeric/factual match": result["numeric_score"]["passed"],
} for result in evaluation_results])
display(score_table)
numeric_accuracy = score_table["numeric/factual match"].mean()
print(f"Numeric/factual accuracy: {numeric_accuracy:.1%} ({score_table['numeric/factual match'].sum()}/{len(score_table)})")


INFO:httpx:HTTP Request: GET https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash-lite "HTTP/1.1 200 OK"
HTTP Request: GET https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash-lite "HTTP/1.1 200 OK"
INFO:google_genai.chats:AFC is enabled with max remote calls: 10.
AFC is enabled with max remote calls: 10.
INFO:google_genai.chats:AFC is enabled with max remote calls: 10.
AFC is enabled with max remote calls: 10.
INFO:google_genai.chats:AFC is enabled with max remote calls: 10.
AFC is enabled with max remote calls: 10.
INFO:google_genai.chats:AFC is enabled with max remote calls: 10.
AFC is enabled with max remote calls: 10.
INFO:google_genai.chats:AFC is enabled with max remote calls: 10.
AFC is enabled with max remote calls: 10.
INFO:google_genai.chats:AFC is enabled with max remote calls: 10.
AFC is enabled with max remote calls: 10.


,id,question,answer,ground truth,numeric/factual match
0,top_gross_customer,Which identified customer generated the most gross sales revenue before returns?,18102,18102,True
1,average_net_clv,What was the average net customer lifetime value for identified customers?,1904.679117955738,"1,950.56",False
2,top_country_revenue,Which country generated the most net revenue across all transaction lines?,United Kingdom,United Kingdom,True
3,largest_line_revenue,What was the largest single positive line revenue?,25111.09,"25,111.09",True
4,identified_customer_count,How many identified customers made at least one positive sale?,4312,"4,312",True
5,total_net_revenue,"What was total net revenue across all transaction lines, including returns?",9539484.634,"9,539,484.63",True


Numeric/factual accuracy: 83.3% (5/6)


## Gemini as judge

One compact request judges all responses, preserving API budget. Correctness asks whether the answer is right; faithfulness asks whether its claims are supported by the frozen ground truth. Treat the judge as a second signal, not as a replacement for numeric matching.

In [5]:
RUN_LLM_JUDGE = True

if RUN_LLM_JUDGE:
    judge_llm = build_gemini_llm(
        model=os.getenv("GEMINI_CHAT_MODEL", "gemini-3.5-flash-lite"),
        temperature=0.0,
        max_tokens=768,
    )
    judge_response = str(await judge_llm.acomplete(build_judge_prompt(evaluation_results)))
    judge_results = parse_judge_response(
        judge_response,
        expected_ids=[item["id"] for item in gold_set],
    )
    judge_table = pd.DataFrame(judge_results)
    final_results = score_table.merge(judge_table, on="id", validate="one_to_one")
    display(final_results)
    print(f"Judge correctness: {final_results['correctness'].mean():.1%}")
    print(f"Judge faithfulness: {final_results['faithfulness'].mean():.1%}")
else:
    final_results = score_table.copy()
    print("LLM judge skipped; numeric/factual scores are available above.")


INFO:httpx:HTTP Request: GET https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash-lite "HTTP/1.1 200 OK"
HTTP Request: GET https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash-lite "HTTP/1.1 200 OK"
INFO:google_genai.chats:AFC is enabled with max remote calls: 10.
AFC is enabled with max remote calls: 10.


,id,question,answer,ground truth,numeric/factual match,correctness,faithfulness,rationale
0,top_gross_customer,Which identified customer generated the most gross sales revenue before returns?,18102,18102,True,True,True,The system answer matches the expected customer ID.
1,average_net_clv,What was the average net customer lifetime value for identified customers?,1904.679117955738,"1,950.56",False,False,False,The system answer does not match the expected average net CLV and numeric_match is false.
2,top_country_revenue,Which country generated the most net revenue across all transaction lines?,United Kingdom,United Kingdom,True,True,True,The system answer correctly identifies the country with the most net revenue.
3,largest_line_revenue,What was the largest single positive line revenue?,25111.09,"25,111.09",True,True,True,The system answer correctly identifies the maximum single line revenue.
4,identified_customer_count,How many identified customers made at least one positive sale?,4312,"4,312",True,True,True,The system answer correctly matches the count of identified customers.
5,total_net_revenue,"What was total net revenue across all transaction lines, including returns?",9539484.634,"9,539,484.63",True,True,True,The system answer matches the expected total net revenue.


Judge correctness: 83.3%
Judge faithfulness: 83.3%


## Interpreting failures

When a response misses the gold target, inspect the benchmark notes alongside the generated Pandas operation (available in the query response metadata). Common causes are mixing gross sales with net CLV, counting guests as customers, using invoice totals instead of line revenue, and omitting returns from net revenue. The judge rationale helps explain the answer but does not establish ground truth.